# KARMIC SEED ASSIGNMENT

# XYZ Fulfillment Hub — Operational Solution
**Role:** Operations Analyst Take-Home Project | **Company:** Karmic Seed
**Author:** ARUSHI MARKAND

## Executive Summary & Core Focus
To solve XYZ’s fulfillment bottlenecks, this application addresses **4 key operational priorities**:
1. **Priority SLA Management:** Automatic visual highlighting and forced top-of-queue sorting for high-priority orders to prevent missed cutoffs.
2. **Low-Friction Warehouse UI:** Simple dropdown filtering and color-coded statuses designed specifically for non-technical floor operators.
3. **Stock Replenishment Triggers:** Automated detection of Main Warehouse stock depletion with auto-calculated transfer quantities from Overflow storage.
4. **Courier & Dispatch Visibility:** Pickup cutoff monitoring aligned with courier reliability metrics.

# IMPORTING REQUIERED LIBRARIES AND DATASETS

In [1]:
import pandas as pd
import numpy as np
import os
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

# Define folder path with a trailing slash
folder_path = r'C:\Users\arush\OneDrive\Desktop\Karmic Seed Data\\'

# Load local CSV files
orders_df = pd.read_csv(os.path.join(folder_path, 'orders.csv'))
inventory_df = pd.read_csv(os.path.join(folder_path, 'inventory.csv'))
products_df = pd.read_csv(os.path.join(folder_path, 'products.csv'))
couriers_df = pd.read_csv(os.path.join(folder_path, 'couriers.csv'))

# Merge master table for operational view
master_df = orders_df.merge(products_df, on='SKU', how='left').merge(inventory_df, on='SKU', how='left')

print("✅ Data successfully loaded!")
display(master_df.head(3))

✅ Data successfully loaded!


,Order_ID,Order_Date,Channel,Customer,Priority,SKU,Quantity,Order_Status,Order_Deadline,Created_Time,...,Courier,Product_Name,Category,Variant,Unit_Price,Main_Warehouse_Stock,Secondary_Warehouse_Stock,Reorder_Level,Physical_Count,Inventory_Discrepancy
0,ORD-1001,2026-09-26,Amazon,Customer 100,Regular,SKU-107,2,Shipped,2026-09-27 12:00,2026-09-26 09:16,...,DTDC,Laptop Stand,Office,Silver,1899,4,30,10,4,0
1,ORD-1002,2026-09-26,Flipkart,Customer 022,Priority,SKU-108,1,Shipped,2026-09-26 12:30,2026-09-26 13:13,...,Delhivery,Webcam 1080p,Electronics,Black,2499,21,8,8,19,2
2,ORD-1003,2026-09-26,Shopify,Customer 108,Regular,SKU-112,1,Packing,2026-09-27 10:00,2026-09-26 08:28,...,Ecom Express,Portable SSD 1TB,Storage,Black,6999,3,12,5,2,1


# Key Operational Metrics & Priority SLA Barometer

In [2]:
def display_kpis(df):
    total_orders = len(df)
    priority_orders = len(df[df['Priority'] == 'Priority'])
    pending_fulfillment = len(df[df['Order_Status'] != 'Shipped'])
    stock_alerts = len(inventory_df[inventory_df['Main_Warehouse_Stock'] <= inventory_df['Reorder_Level']])
    
    html_code = f"""
    <div style="display: flex; justify-content: space-between; margin-bottom: 20px;">
        <div style="background-color: #f8f9fa; padding: 15px; border-radius: 8px; width: 22%; border-left: 5px solid #007bff; box-shadow: 2px 2px 5px #ccc;">
            <h4 style="margin:0; color:#555;">Total Orders</h4>
            <h2 style="margin:5px 0 0 0;">{total_orders}</h2>
        </div>
        <div style="background-color: #fff3cd; padding: 15px; border-radius: 8px; width: 22%; border-left: 5px solid #ffc107; box-shadow: 2px 2px 5px #ccc;">
            <h4 style="margin:0; color:#555;">Priority SLA Queue</h4>
            <h2 style="margin:5px 0 0 0; color:#856404;">{priority_orders}</h2>
        </div>
        <div style="background-color: #f8d7da; padding: 15px; border-radius: 8px; width: 22%; border-left: 5px solid #dc3545; box-shadow: 2px 2px 5px #ccc;">
            <h4 style="margin:0; color:#555;">Pending Fulfillment</h4>
            <h2 style="margin:5px 0 0 0; color:#721c24;">{pending_fulfillment}</h2>
        </div>
        <div style="background-color: #e2e3e5; padding: 15px; border-radius: 8px; width: 22%; border-left: 5px solid #6c757d; box-shadow: 2px 2px 5px #ccc;">
            <h4 style="margin:0; color:#555;">Stock Replenishment Triggers</h4>
            <h2 style="margin:5px 0 0 0;">{stock_alerts}</h2>
        </div>
    </div>
    """
    display(HTML(html_code))

display_kpis(master_df)

# Interactive Floor Terminal

In [3]:
status_filter = widgets.Dropdown(
    options=['All Active Queue'] + list(master_df['Order_Status'].unique()),
    value='All Active Queue',
    description='Status Filter:',
    disabled=False,
)

output_area = widgets.Output()

def filter_orders(change):
    with output_area:
        clear_output()
        selected = change['new'] if isinstance(change, dict) else status_filter.value
        
        if selected == 'All Active Queue':
            filtered = master_df[master_df['Order_Status'] != 'Shipped'].copy()
        else:
            filtered = master_df[master_df['Order_Status'] == selected].copy()
            
        # Priority sorting: Priority orders first, then by Order Deadline
        filtered['Priority_Rank'] = filtered['Priority'].apply(lambda x: 0 if x == 'Priority' else 1)
        filtered = filtered.sort_values(by=['Priority_Rank', 'Order_Deadline'])
        
        display(HTML(f"<h3>📦 Warehouse Floor Execution Queue ({len(filtered)} Orders)</h3>"))
        
        cols_to_show = ['Order_ID', 'Priority', 'Product_Name', 'Variant', 'Quantity', 'Main_Warehouse_Stock', 'Order_Status', 'Order_Deadline', 'Courier']
        
        # Color coding priority rows using .applymap() for backwards compatibility
        styled_df = filtered[cols_to_show].style.applymap(
            lambda x: 'background-color: #ffcccc; font-weight: bold; color: #900;' if x == 'Priority' else '', subset=['Priority']
        )
        display(styled_df)

status_filter.observe(filter_orders, names='value')
display(status_filter)
display(output_area)
filter_orders(None)

Dropdown(description='Status Filter:', options=('All Active Queue', 'Shipped', 'Packing', 'Received', 'Picking…

Output()

# Inventory Replenishment & Stock Transfer Engine

In [4]:
display(HTML("<h3>🏭 Inventory Replenishment & Stock Transfer Engine</h3>"))

low_stock_df = inventory_df[inventory_df['Main_Warehouse_Stock'] <= inventory_df['Reorder_Level']].copy()
low_stock_df['Transfer_Qty_Needed'] = (low_stock_df['Reorder_Level'] * 2) - low_stock_df['Main_Warehouse_Stock']

if not low_stock_df.empty:
    display(HTML("<p style='color:red;'><b>⚠️ Low stock detected in Main Warehouse! Stock must be moved from Secondary Warehouse before picking:</b></p>"))
    display(low_stock_df[['SKU', 'Main_Warehouse_Stock', 'Secondary_Warehouse_Stock', 'Reorder_Level', 'Transfer_Qty_Needed']])
else:
    display(HTML("<p style='color:green;'><b>✅ Main warehouse stock levels are healthy. No transfers required.</b></p>"))

,SKU,Main_Warehouse_Stock,Secondary_Warehouse_Stock,Reorder_Level,Transfer_Qty_Needed
2,SKU-103,7,35,12,17
4,SKU-105,8,25,8,8
6,SKU-107,4,30,10,16
10,SKU-111,9,28,10,11
11,SKU-112,3,12,5,7
14,SKU-115,6,19,7,8


# COURIER AND DISPATCH

In [5]:
display(HTML("<h3>🚚 Courier & Dispatch Performance Monitor</h3>"))

# Merge master queue with courier metadata
courier_summary = master_df[master_df['Order_Status'] != 'Shipped'].groupby('Courier').agg(
    Pending_Orders=('Order_ID', 'count'),
    Priority_Pending=('Priority', lambda x: (x == 'Priority').sum())
).reset_index()

courier_dispatch_df = courier_summary.merge(couriers_df, on='Courier', how='left')

# Display courier dispatch table with all available courier columns
display(courier_dispatch_df)

,Courier,Pending_Orders,Priority_Pending,Cost,Expected_Delivery_Days,Pickup_Time,Reliability
0,Blue Dart,44,9,110,1,17:00,0.97
1,DTDC,40,9,70,2,16:30,0.92
2,Delhivery,52,14,75,2,16:00,0.94
3,Ecom Express,34,5,65,3,15:30,0.91
